# Inspect paddocks

Once a pipeline run has produced a paddocks GeoPackage (either via SAM
segmentation or your own polygons), you can load it as a GeoDataFrame
and ask the obvious questions: how many paddocks, how big, how compact,
how does each one's NDVI behave?

This notebook walks through a typical inspection workflow.

> **Prerequisite.** You need a finished pipeline run for the `Query`
> below. If you haven't run one yet, do [`02_pipeline_stages.ipynb`](02_pipeline_stages.ipynb)
> first (it uses `stub="stages_demo"`) — or edit the cell below to point
> at any other run you have on disk.


In [ ]:
from datetime import date
from pathlib import Path
from PaddockTS.query import Query

# Edit these to match a pipeline run you've completed.
query = Query(
    bbox=[148.36265, -33.52606, 148.38265, -33.50606],
    start=date(2024, 1, 1),
    end=date(2024, 12, 31),
    stub="stages_demo",
)
print(query)
print("paddocks file:", query.sam_paddocks_path)


## 1. Load the paddocks GeoDataFrame

The GeoPackage has one row per paddock with columns:

- `geometry` — paddock polygon (in EPSG:6933 by default)
- `paddock` — 1-based integer ID
- `area_ha` — area in hectares (computed in a local UTM zone)
- `compactness` — isoperimetric `4πA / L²` ∈ [0, 1]; 1 is a circle


In [ ]:
import geopandas as gpd

paddocks = gpd.read_file(query.sam_paddocks_path)
print(f"{len(paddocks)} paddocks, CRS={paddocks.crs}")
paddocks.head()


## 2. Summary statistics


In [ ]:
paddocks[["area_ha", "compactness"]].describe()


## 3. Map them — coloured by area


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 9))
paddocks.plot(column="area_ha", cmap="viridis", legend=True,
              legend_kwds={"label": "area (ha)", "shrink": 0.6}, ax=ax)
paddocks.boundary.plot(ax=ax, color="black", linewidth=0.4)
ax.set_title(f"{len(paddocks)} paddocks coloured by area")
ax.set_aspect("equal")
plt.show()


## 4. Filter

Drop small or sliver-shaped paddocks for focused downstream analysis:


In [ ]:
big      = paddocks[paddocks.area_ha > 20]
compact  = paddocks[paddocks.compactness > 0.5]
big_and_compact = paddocks[(paddocks.area_ha > 20) & (paddocks.compactness > 0.5)]

print(f"   area > 20 ha:                 {len(big):>3}")
print(f"   compactness > 0.5:            {len(compact):>3}")
print(f"   both:                         {len(big_and_compact):>3}")


## 5. Join with per-paddock NDVI

The pipeline stores per-paddock medians of every band + index in
`sam_paddocks_timeseries.zarr`. Compute the mean NDVI over the entire
window per paddock and join it back to the GeoDataFrame for thematic
mapping.


In [ ]:
import xarray as xr

ts_path = f"{query.tmp_dir}/sam_paddocks_timeseries.zarr"
ts = xr.open_zarr(ts_path, decode_coords="all")

ndvi_mean = ts.NDVI.mean(dim="time").to_pandas()
print(ndvi_mean.head())

# Paddock IDs in the TS are strings; cast for the join.
paddocks["paddock_str"] = paddocks["paddock"].astype(str)
paddocks["ndvi_mean"]   = paddocks["paddock_str"].map(ndvi_mean.to_dict())
paddocks[["paddock", "area_ha", "ndvi_mean"]].head()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
paddocks.plot(column="ndvi_mean", cmap="RdYlGn", vmin=0, vmax=0.8,
              legend=True, legend_kwds={"label": "mean NDVI", "shrink": 0.6}, ax=ax)
paddocks.boundary.plot(ax=ax, color="black", linewidth=0.4)
ax.set_title("Paddocks coloured by mean NDVI over the run window")
ax.set_aspect("equal")
plt.show()


## See also

- [`05_inspect_phenology.ipynb`](05_inspect_phenology.ipynb) — per-paddock seasonal metrics.
- [`06_inspect_time_series.ipynb`](06_inspect_time_series.ipynb) — full per-paddock × time × variable cube.
- [`07_inspect_videos.ipynb`](07_inspect_videos.ipynb) — Sentinel-2 / fractional-cover MP4s with paddock overlays.
